In [38]:
import matplotlib.pyplot as plt
import numpy as np

from ase.build import bulk
from ase.calculators.emt import EMT
from ase.eos import EquationOfState
from ase.filters import FrechetCellFilter
from ase.io.espresso import write_espresso_in
from ase.optimize import BFGS
from ase.units import kJ
from ase.visualize.plot import plot_atoms

from ase.visualize import view
from ase import Atoms

In [39]:
atoms = bulk('Ag')

In [41]:
view(atoms * (3,3,3), viewer="x3d")

In [42]:
def make_diamond_slab(symbol, a, n_layers):
    """
    Returns a [001] diamond-cubic slab with n_layers unit-cell layers
    """
    uc = bulk(symbol, crystalstructure="diamond", a=a, cubic=True)
    slab = uc.repeat((1,1,n_layers))
    return slab


In [43]:
KPTS = (4,4,1)
QPTS = (2,2,1)

In [44]:
A_SI = 5.431
N_SI = 4
N_GE = 4

def build_interface():
    si_slab = make_diamond_slab("Si", A_SI, N_SI)
    ge_slab = make_diamond_slab("Ge", A_SI, N_GE)

    z_shift = si_slab.get_cell()[2,2]
    ge_pos = ge_slab.get_positions()
    si_pos = si_slab.get_positions() + np.array([0, 0, z_shift])

    all_pos = np.vstack([ge_pos, si_pos])
    all_symbols = list(ge_slab.get_chemical_symbols()) + list(si_slab.get_chemical_symbols())

    total_z = z_shift + ge_slab.get_cell()[2,2]
    cell = si_slab.get_cell().copy()
    cell[2,2] = total_z

    interface = Atoms(
        symbols=all_symbols,
        cell=cell,
        positions=all_pos,
        pbc=[True, True, True],
    )

    interface.center(axis=2)
    return interface

In [45]:
interface = build_interface()

In [70]:
view(interface, viewer="x3d")

In [50]:
from ase.io.espresso import write_espresso_in

In [54]:
Pseudopotentials = {
    'Si': "Si.pbe-n-rrkjus_psl.1.0.0.UPF",
    'Ge': "ge_pbe_v1.4.uspp.F.UPF",
}

input_data = {
    'calculation': 'relax',
    'pseudo_dir': '/usr/scratch/mont-fort11/pfischill/pseudo',
    'restart_mode': 'from_scratch',
    'tprnfor': True,
    'etot_conv_thr': 1e-5,
    'forc_conv_thr': 1e-4,
    'ecutwfc': 60,
    'ecutrho': 480,
    'input_dft': 'rpbe',
    'vdw_corr': 'dft-d3',
    'occupations': 'smearing',
    'degauss': 0.01,
    'smearing': 'cold',
    'conv_thr': 1e-8,
    'mixing_mode': 'local-TF',
    'mixing_beta': 0.35,
    'diagonalization': 'david',
    'ion_dynamics': 'bfgs',
    'bfgs_ndim': 6,
    'startingwfc': 'random',
}
write_espresso_in('pw.in', interface, pseudopotentials=Pseudopotentials,
                   input_data=input_data, format='espresso-in')

In [55]:
from ase.io.espresso import read_espresso_out

In [62]:
with open('pw.out') as f:
    results = list(read_espresso_out(f))

In [64]:
view(results[1], viewer='x3d')

In [69]:
from ase.io.espresso import write_espresso_ph, read_espresso_ph
ph_input_data = {
  'tr2_ph': 1.0e-12,
  'prefix': 'pwscf',
  'verbosity': 'high',
  'ldisp': True,
  'qplot': False,
  'nq1': QPTS[0],
  'nq2': QPTS[1],
  'nq3': QPTS[2],
  'alpha_mix(1)': 0.1,
}

with open('ph.in', mode='w') as f:
    write_espresso_ph(f, ph_input_data, qpts=QPTS)